# 12 - Visualization of completed experiment results

This notebook is a post-hoc visualization layer for notebooks 01-11. It reads exported CSV artifacts only. It does not retrain models, refit rules, change thresholds, or select a new policy. All figures use an explicit color palette and are exported as RGB PNG and PDF files.

In [1]:
from pathlib import Path
import os
import subprocess
import sys
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

REPO_URL = 'https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git'
KAGGLE = Path('/kaggle').exists()
KAGGLE_REPO = Path('/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection')

REQUIRED_ARTIFACTS = {
    'predictive': 'table_cross_dataset_predictive_test.csv',
    'explanation': 'table_cross_dataset_explanation_quality.csv',
    'rules': 'table_cross_dataset_rule_quality.csv',
    'calibration': 'table_raw_vs_calibrated_probability_metrics.csv',
    'ablation': 'table_ieee_rule_ablation.csv',
    'stress': 'primary_stress_test_synthesis.csv',
}

def find_artifact(filename):
    roots = [Path.cwd(), *Path.cwd().parents, Path('/kaggle/input'), Path('/kaggle/working')]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(root.rglob(filename))
    matches = [p for p in matches if p.is_file()]
    if not matches:
        return None
    # Prefer a canonical results tree, then the shortest mounted path.
    matches.sort(key=lambda p: (0 if 'results' in p.parts else 1, len(p.parts), str(p)))
    return matches[0]

ARTIFACTS = {key: find_artifact(filename) for key, filename in REQUIRED_ARTIFACTS.items()}
missing = [name for name, path in ARTIFACTS.items() if path is None]
if missing:
    searched = [str(Path('/kaggle/input')), str(Path('/kaggle/working')), str(Path.cwd())]
    raise FileNotFoundError(f'Missing visualization artifacts: {missing}. Searched: {searched}. Add the CSV outputs from notebooks 08 and 11 as Kaggle Input.')

PROJECT_ROOT = Path.cwd()
RESULTS = PROJECT_ROOT / 'results'
CLASSIC = ARTIFACTS['predictive'].parent
STRESS = ARTIFACTS['stress'].parent
OUTPUT_ROOT = Path('/kaggle/working/visualization_results') if KAGGLE else RESULTS / 'figures'
FIG_DIR = OUTPUT_ROOT / 'visualization'
FIG_DIR.mkdir(parents=True, exist_ok=True)

PALETTE = {
    'IEEE-CIS': '#2F6BFF',
    'BAF': '#1B9E77',
    'TransXion v2': '#E68613',
    'AMLNet v1.0': '#6F4CC3',
    'MLP': '#D55E00',
    'TabularResNet': '#2F6BFF',
    'LightGBM': '#1B9E77',
    'xgboost': '#E68613',
    '10%': '#2F6BFF',
    '25%': '#E68613',
}

plt.rcParams.update({'figure.dpi': 140, 'savefig.dpi': 300, 'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 10, 'pdf.fonttype': 42, 'ps.fonttype': 42})
print('Project root:', PROJECT_ROOT)
print('Classic results:', CLASSIC)
print('Stress results:', STRESS)
print('Figure output:', FIG_DIR)

Project root: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection
Classic results: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\runs\notebooks
Stress results: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\stress\notebooks
Figure output: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization


In [2]:
def read_csv(path):
    path = Path(path)
    if not path.exists():
        print('Missing artifact:', path)
        return pd.DataFrame()
    frame = pd.read_csv(path)
    frame['_source_path'] = str(path)
    return frame

def save_figure(fig, stem):
    png = FIG_DIR / f'{stem}.png'
    pdf = FIG_DIR / f'{stem}.pdf'
    fig.savefig(png, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    print('saved:', png)
    print('saved:', pdf)

def color_for(value, fallback='#58606B'):
    text = str(value).lower()
    for key, color in PALETTE.items():
        if key.lower() in text:
            return color
    return fallback

predictive = read_csv(ARTIFACTS['predictive'])
explanation = read_csv(ARTIFACTS['explanation'])
rules = read_csv(ARTIFACTS['rules'])
raw_calibration = read_csv(ARTIFACTS['calibration'])
ablation = read_csv(ARTIFACTS['ablation'])
stress = read_csv(ARTIFACTS['stress'])

display(pd.DataFrame({
    'artifact': ['predictive', 'explanation', 'rules', 'calibration', 'ablation', 'stress'],
    'rows': [len(predictive), len(explanation), len(rules), len(raw_calibration), len(ablation), len(stress)],
}))

,artifact,rows
0,predictive,6
1,explanation,2
2,rules,25
3,calibration,4
4,ablation,16
5,stress,4


## 1. Predictive performance across IEEE-CIS and BAF

In [3]:
if predictive.empty:
    print('Skipped: predictive synthesis table is unavailable.')
else:
    d = predictive[predictive['split'].eq('test')].copy()
    d['model_label'] = d['model'].astype(str).str.replace('xgboost', 'XGBoost', regex=False)
    for metric, label in [('pr_auc_mean', 'AUPRC'), ('roc_auc_mean', 'AUROC'), ('fbeta_mean', 'F2')]:
        if metric not in d:
            continue
        d[metric] = pd.to_numeric(d[metric], errors='coerce')
        d = d.dropna(subset=[metric])
        models = list(d['model_label'].drop_duplicates())
        datasets = list(d['dataset'].drop_duplicates())
        x = np.arange(len(models))
        width = 0.78 / max(1, len(datasets))
        fig, ax = plt.subplots(figsize=(10.5, 5.0))
        for j, dataset in enumerate(datasets):
            part = d[d['dataset'].eq(dataset)].set_index('model_label').reindex(models)
            vals = part[metric].to_numpy(dtype=float)
            off = (j - (len(datasets)-1)/2) * width
            ax.bar(x + off, vals, width=width, color=color_for(dataset), label=dataset, alpha=0.9)
        ax.set_xticks(x)
        ax.set_xticklabels(models, rotation=25, ha='right')
        ax.set_ylabel(label)
        ax.set_title(f'Locked-test {label} by dataset and predictor')
        ax.grid(axis='y', alpha=0.25)
        ax.legend(frameon=False)
        fig.tight_layout()
        save_figure(fig, f'predictive_{metric.replace("_mean", "")}')
        plt.show()

saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\predictive_pr_auc.png
saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\predictive_pr_auc.pdf


C:\Users\LEGION\AppData\Local\Temp\ipykernel_5936\3588302322.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\predictive_roc_auc.png
saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\predictive_roc_auc.pdf


C:\Users\LEGION\AppData\Local\Temp\ipykernel_5936\3588302322.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\predictive_fbeta.png
saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\predictive_fbeta.pdf


C:\Users\LEGION\AppData\Local\Temp\ipykernel_5936\3588302322.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Calibration and explanation quality

In [4]:
if not raw_calibration.empty:
    cal = raw_calibration.copy()
    for metric in ['brier', 'ece']:
        if metric not in cal:
            continue
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        for probability_type, part in cal.groupby('probability_type'):
            ax.bar(part['dataset'].astype(str) + ' / ' + probability_type, part[metric], color=color_for(probability_type), alpha=0.9, label=probability_type)
        ax.set_ylabel(metric.upper())
        ax.set_title(f'{metric.upper()} before and after calibration')
        ax.tick_params(axis='x', rotation=35)
        ax.grid(axis='y', alpha=0.25)
        fig.tight_layout()
        save_figure(fig, f'calibration_{metric}')
        plt.show()

if not explanation.empty:
    e = explanation.copy()
    for column in ['explained_alert_precision', 'all_alert_precision', 'explained_alert_precision_gain']:
        e[column] = pd.to_numeric(e[column], errors='coerce')
    x = np.arange(len(e))
    width = 0.36
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    ax.bar(x - width/2, e['all_alert_precision'], width, label='All alerts', color='#A7B0BA')
    ax.bar(x + width/2, e['explained_alert_precision'], width, label='Explained alerts', color='#2F6BFF')
    ax.set_xticks(x)
    ax.set_xticklabels(e['dataset'].astype(str))
    ax.set_ylabel('Precision')
    ax.set_title('Explanation precision versus the full alert queue')
    ax.grid(axis='y', alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    save_figure(fig, 'explanation_precision_comparison')
    plt.show()

saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\calibration_brier.png
saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\calibration_brier.pdf
saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\calibration_ece.png
saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\calibration_ece.pdf


C:\Users\LEGION\AppData\Local\Temp\ipykernel_5936\659923196.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\LEGION\AppData\Local\Temp\ipykernel_5936\659923196.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\explanation_precision_comparison.png
saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\explanation_precision_comparison.pdf


C:\Users\LEGION\AppData\Local\Temp\ipykernel_5936\659923196.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Rule quality, ablation, and stress-test guardrails

In [5]:
if not rules.empty:
    r = rules.copy()
    r = r[r['split'].eq('train')].copy() if 'split' in r else r
    r['lift'] = pd.to_numeric(r['lift'], errors='coerce')
    r = r.dropna(subset=['lift']).sort_values('lift', ascending=False).head(20)
    labels = r['dataset'].astype(str) + ' / ' + r['rule'].astype(str)
    fig, ax = plt.subplots(figsize=(11, 5.2))
    ax.bar(np.arange(len(r)), r['lift'], color=[color_for(x) for x in r['dataset']], alpha=0.9)
    ax.set_xticks(np.arange(len(r)))
    ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=8)
    ax.set_ylabel('Lift')
    ax.set_title('Audited rule lift with dataset provenance')
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    save_figure(fig, 'rule_lift_by_dataset')
    plt.show()

if not stress.empty:
    s = stress.copy()
    s['dataset_label'] = s['dataset'].map({'transxion_v2': 'TransXion v2', 'amlnet_v1_0': 'AMLNet v1.0'}).fillna(s['dataset'])
    s['coverage_label'] = (pd.to_numeric(s['coverage_budget'], errors='coerce') * 100).astype(int).astype(str) + '%'
    s['support_rate'] = pd.to_numeric(s['supported_alert_rate'], errors='coerce')
    s['score_precision'] = pd.to_numeric(s['score_only_requested_budget_precision'], errors='coerce')
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.7))
    for dataset, part in s.groupby('dataset_label', sort=False):
        axes[0].plot(part['coverage_label'], part['support_rate'], marker='o', linewidth=2.2, label=dataset, color=color_for(dataset))
        axes[1].plot(part['coverage_label'], part['score_precision'], marker='o', linewidth=2.2, label=dataset, color=color_for(dataset))
    axes[0].set_title('Rule support on stress-test alerts')
    axes[0].set_ylabel('Supported-alert rate')
    axes[1].set_title('Score-only precision at requested budget')
    axes[1].set_ylabel('Precision')
    for ax in axes:
        ax.set_xlabel('Requested coverage')
        ax.grid(alpha=0.25)
        ax.legend(frameon=False)
    fig.tight_layout()
    save_figure(fig, 'stress_support_and_score_only')
    plt.show()

if not ablation.empty and 'precision_gain' in ablation:
    a = ablation.copy()
    a['precision_gain'] = pd.to_numeric(a['precision_gain'], errors='coerce')
    a = a.dropna(subset=['precision_gain'])
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.bar(a['ablation'].astype(str), a['precision_gain'], color='#1B9E77', alpha=0.9)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Precision gain')
    ax.set_title('IEEE-CIS rule ablation precision gain')
    ax.tick_params(axis='x', rotation=35)
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    save_figure(fig, 'ieee_rule_ablation_precision_gain')
    plt.show()

saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\rule_lift_by_dataset.png
saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\rule_lift_by_dataset.pdf


C:\Users\LEGION\AppData\Local\Temp\ipykernel_5936\1903734865.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\stress_support_and_score_only.png
saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\stress_support_and_score_only.pdf


C:\Users\LEGION\AppData\Local\Temp\ipykernel_5936\1903734865.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\ieee_rule_ablation_precision_gain.png
saved: E:\Study\Thực tập tốt nghiệp\Explainable_NeuroSymbolic_Fraud_Detection\results\figures\visualization\ieee_rule_ablation_precision_gain.pdf


C:\Users\LEGION\AppData\Local\Temp\ipykernel_5936\1903734865.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
files = sorted([p for p in FIG_DIR.iterdir() if p.suffix.lower() in {'.png', '.pdf'}])
manifest = pd.DataFrame({'figure': [p.name for p in files], 'path': [str(p) for p in files], 'bytes': [p.stat().st_size for p in files]})
manifest.to_csv(FIG_DIR / 'visualization_manifest.csv', index=False)
display(manifest)
print('Done. Figures were generated from frozen CSV artifacts with explicit colors.')

,figure,path,bytes
0,calibration_brier.pdf,E:\Study\Thực tập tốt nghiệp\Explainable_Neuro...,11457
1,calibration_brier.png,E:\Study\Thực tập tốt nghiệp\Explainable_Neuro...,83764
2,calibration_ece.pdf,E:\Study\Thực tập tốt nghiệp\Explainable_Neuro...,11570
3,calibration_ece.png,E:\Study\Thực tập tốt nghiệp\Explainable_Neuro...,86110
4,explanation_precision_comparison.pdf,E:\Study\Thực tập tốt nghiệp\Explainable_Neuro...,11816
5,explanation_precision_comparison.png,E:\Study\Thực tập tốt nghiệp\Explainable_Neuro...,64977
6,ieee_rule_ablation_precision_gain.pdf,E:\Study\Thực tập tốt nghiệp\Explainable_Neuro...,13554
7,ieee_rule_ablation_precision_gain.png,E:\Study\Thực tập tốt nghiệp\Explainable_Neuro...,237573
8,predictive_fbeta.pdf,E:\Study\Thực tập tốt nghiệp\Explainable_Neuro...,12608
9,predictive_fbeta.png,E:\Study\Thực tập tốt nghiệp\Explainable_Neuro...,82162


Done. Figures were generated from frozen CSV artifacts with explicit colors.
